# Model Experiments
Phase 4 baseline: reproducible random split and Linear Regression.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import time
import pandas as pd
from sklearn.linear_model import LinearRegression
from src.evaluation.metrics import evaluate_all
from src.models.train import (

    get_features_and_target,

    make_train_val_split,

    train_random_forest,

    train_xgboost,

    train_lightgbm,

    generate_oof_predictions,

)

In [3]:
data_path = PROJECT_ROOT / 'data' / 'processed' / 'train_features.parquet'
df = pd.read_parquet(data_path)
X, y = get_features_and_target(df)
X_train, X_val, y_train, y_val = make_train_val_split(X, y)
print(f'X: {X.shape}; y: {y.shape}')
print(X.columns.tolist())

X: (1429225, 10); y: (1429225,)
['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'distance_km', 'pickup_hour', 'pickup_dayofweek', 'is_weekend']


In [4]:
start = time.perf_counter()
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
train_time = time.perf_counter() - start
val_predictions = lr_model.predict(X_val)
results = evaluate_all(y_val.to_numpy(), val_predictions)
results['train_time_sec'] = round(train_time, 2)
print('Linear Regression baseline:')
for metric, value in results.items():
    print(f'  {metric}: {value:.4f}' if isinstance(value, float) else f'  {metric}: {value}')

Linear Regression baseline:
  rmsle: 0.5086
  rmse: 568.8683
  mae: 277.2094
  train_time_sec: 0.2400


## Random Forest



First verify the training function on a small sample, then run the full training cell once.

In [5]:
# Fast sanity check only; this is not the final Random Forest result.

sample_indices = X_train.sample(20_000, random_state=42).index

sample_model, sample_time = train_random_forest(

    X_train.loc[sample_indices], y_train.loc[sample_indices]

)

print(f'Sample fit time: {sample_time:.1f} seconds')

Sample fit time: 1.1 seconds


In [6]:
# Full training run: run once after the sample cell succeeds.

rf_model, rf_train_time = train_random_forest(X_train, y_train)

rf_val_predictions = rf_model.predict(X_val)

rf_results = evaluate_all(y_val.to_numpy(), rf_val_predictions)

rf_results['train_time_sec'] = round(rf_train_time, 2)



print('Random Forest:')

for metric, value in rf_results.items():

    print(f'  {metric}: {value:.4f}' if isinstance(value, float) else f'  {metric}: {value}')

Random Forest:
  rmsle: 0.3747
  rmse: 514.7983
  mae: 203.8764
  train_time_sec: 144.7200


In [7]:
comparison = pd.DataFrame([

    {'model': 'Linear Regression', **results},

    {'model': 'Random Forest', **rf_results},

])

comparison

,model,rmsle,rmse,mae,train_time_sec
0,Linear Regression,0.508631,568.868339,277.209412,0.24
1,Random Forest,0.374704,514.798326,203.876356,144.72


## XGBoost



Verify XGBoost on a small sample before committing to the full training run.

In [8]:
# Fast sanity check only; this is not the final XGBoost result.

sample_indices = X_train.sample(20_000, random_state=42).index

sample_xgb_model, sample_xgb_time = train_xgboost(

    X_train.loc[sample_indices], y_train.loc[sample_indices], X_val, y_val

)

print(

    f'Sample fit time: {sample_xgb_time:.1f} seconds; '

    f'best iteration: {sample_xgb_model.best_iteration}'

)

Sample fit time: 1.7 seconds; best iteration: 18


In [9]:
# Full training run: run once after the sample cell succeeds.

xgb_model, xgb_train_time = train_xgboost(X_train, y_train, X_val, y_val)

xgb_val_predictions = xgb_model.predict(X_val)

xgb_results = evaluate_all(y_val.to_numpy(), xgb_val_predictions)

xgb_results['train_time_sec'] = round(xgb_train_time, 2)



print('XGBoost:')

for metric, value in xgb_results.items():

    print(f'  {metric}: {value:.4f}' if isinstance(value, float) else f'  {metric}: {value}')

print(f'Stopped at iteration: {xgb_model.best_iteration} / {xgb_model.n_estimators}')

XGBoost:
  rmsle: 0.3734
  rmse: 508.5402
  mae: 203.3581
  train_time_sec: 7.6100
Stopped at iteration: 113 / 300


In [10]:
comparison = pd.concat(

    [comparison, pd.DataFrame([{'model': 'XGBoost', **xgb_results}])],

    ignore_index=True,

)

comparison

,model,rmsle,rmse,mae,train_time_sec
0,Linear Regression,0.508631,568.868339,277.209412,0.24
1,Random Forest,0.374704,514.798326,203.876356,144.72
2,XGBoost,0.373390,508.540236,203.358139,7.61


## LightGBM



Verify LightGBM on a small sample before its full training run.

In [11]:
# Fast sanity check only; this is not the final LightGBM result.

sample_indices = X_train.sample(20_000, random_state=42).index

sample_lgbm_model, sample_lgbm_time = train_lightgbm(

    X_train.loc[sample_indices], y_train.loc[sample_indices], X_val, y_val

)

print(

    f'Sample fit time: {sample_lgbm_time:.1f} seconds; '

    f'best iteration: {sample_lgbm_model.best_iteration_}'

)

c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Sample fit time: 0.4 seconds; best iteration: 39


In [12]:
# Full training run: run once after the sample cell succeeds.

lgbm_model, lgbm_train_time = train_lightgbm(X_train, y_train, X_val, y_val)

lgbm_val_predictions = lgbm_model.predict(X_val)

lgbm_results = evaluate_all(y_val.to_numpy(), lgbm_val_predictions)

lgbm_results['train_time_sec'] = round(lgbm_train_time, 2)



print('LightGBM:')

for metric, value in lgbm_results.items():

    print(f'  {metric}: {value:.4f}' if isinstance(value, float) else f'  {metric}: {value}')

print(f'Stopped at iteration: {lgbm_model.best_iteration_} / {lgbm_model.n_estimators}')

c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM:
  rmsle: 0.3694
  rmse: 506.7180
  mae: 200.0079
  train_time_sec: 5.6600
Stopped at iteration: 215 / 300


In [13]:
comparison = pd.concat(

    [comparison, pd.DataFrame([{'model': 'LightGBM', **lgbm_results}])],

    ignore_index=True,

)

comparison

,model,rmsle,rmse,mae,train_time_sec
0,Linear Regression,0.508631,568.868339,277.209412,0.24
1,Random Forest,0.374704,514.798326,203.876356,144.72
2,XGBoost,0.373390,508.540236,203.358139,7.61
3,LightGBM,0.369437,506.717963,200.007910,5.66


## Phase 8 — Comparison and Practical Tuning



Inspect feature importance, test a small set of reasoned configurations, and check where the best model's errors concentrate.

In [14]:
def get_importance_df(model, feature_names, model_name):

    return pd.DataFrame({

        'feature': feature_names,

        'importance': model.feature_importances_,

        'model': model_name,

    })



importances = pd.concat([

    get_importance_df(rf_model, X_train.columns, 'Random Forest'),

    get_importance_df(xgb_model, X_train.columns, 'XGBoost'),

    get_importance_df(lgbm_model, X_train.columns, 'LightGBM'),

], ignore_index=True)

importances['importance_norm'] = importances.groupby('model')['importance'].transform(

    lambda values: values / values.sum()

)

importance_pivot = importances.pivot(

    index='feature', columns='model', values='importance_norm'

)

importance_pivot.sort_values('XGBoost', ascending=False)

model,LightGBM,Random Forest,XGBoost
feature,,,
distance_km,0.128837,0.704111,0.628211
pickup_hour,0.173333,0.071927,0.059868
pickup_longitude,0.139380,0.044630,0.047607
dropoff_longitude,0.170543,0.051806,0.043834
pickup_dayofweek,0.067597,0.018049,0.043088
dropoff_latitude,0.165271,0.061080,0.038119
is_weekend,0.006667,0.008770,0.036923
vendor_id,0.007597,0.000817,0.035881
passenger_count,0.017054,0.005336,0.034670


In [15]:
# Primary selection criterion: the lowest validation RMSLE.

comparison.sort_values('rmsle').reset_index(drop=True)



model_by_name = {

    'Random Forest': rf_model,

    'XGBoost': xgb_model,

    'LightGBM': lgbm_model,

}

predictions_by_name = {

    'Random Forest': rf_val_predictions,

    'XGBoost': xgb_val_predictions,

    'LightGBM': lgbm_val_predictions,

}

best_original_name = comparison.loc[comparison['rmsle'].idxmin(), 'model']

if best_original_name not in model_by_name:

    raise ValueError('A tree model must be the best model before Phase 8 tuning.')

best_original_model = model_by_name[best_original_name]

best_original_predictions = predictions_by_name[best_original_name]

print(f'Best original model by RMSLE: {best_original_name}')

Best original model by RMSLE: LightGBM


In [16]:
# Fast development-mode tuning: all results below use only 200,000 training rows.

sample_indices = X_train.sample(200_000, random_state=42).index

X_sub, y_sub = X_train.loc[sample_indices], y_train.loc[sample_indices]



if best_original_name == 'LightGBM':

    train_best_model = train_lightgbm

    tuning_configs = {

        'baseline_current': dict(n_estimators=300, learning_rate=0.1, num_leaves=31),

        'slower_lr_more_trees': dict(n_estimators=500, learning_rate=0.05, num_leaves=31),

        'more_capacity': dict(n_estimators=300, learning_rate=0.1, num_leaves=63),

    }

elif best_original_name == 'XGBoost':

    train_best_model = train_xgboost

    tuning_configs = {

        'baseline_current': dict(n_estimators=300, learning_rate=0.1, max_depth=6),

        'slower_lr_more_trees': dict(n_estimators=500, learning_rate=0.05, max_depth=6),

        'more_capacity': dict(n_estimators=300, learning_rate=0.1, max_depth=8),

    }

else:

    raise ValueError('Phase 8 targeted tuning supports LightGBM or XGBoost only.')



tuning_results = []

for config_name, params in tuning_configs.items():

    candidate_model, candidate_time = train_best_model(

        X_sub, y_sub, X_val, y_val, **params

    )

    candidate_predictions = candidate_model.predict(X_val)

    candidate_metrics = evaluate_all(y_val.to_numpy(), candidate_predictions)

    candidate_metrics.update({'config': config_name, 'train_time_sec': round(candidate_time, 2)})

    tuning_results.append(candidate_metrics)



tuning_table = pd.DataFrame(tuning_results).sort_values('rmsle').reset_index(drop=True)

tuning_table

c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,rmsle,rmse,mae,config,train_time_sec
0,0.375598,510.334606,203.244504,more_capacity,1.20
1,0.376457,510.326911,204.080587,slower_lr_more_trees,2.32
2,0.376726,510.675090,204.047473,baseline_current,1.68


In [17]:
# Run this once after reviewing tuning_table. It promotes its best subsample configuration.

best_config_name = tuning_table.loc[0, 'config']

best_params = tuning_configs[best_config_name]

final_tuned_model, final_tuned_time = train_best_model(

    X_train, y_train, X_val, y_val, **best_params

)

tuned_predictions = final_tuned_model.predict(X_val)

tuned_results = evaluate_all(y_val.to_numpy(), tuned_predictions)

tuned_results['train_time_sec'] = round(final_tuned_time, 2)

tuned_results['config'] = best_config_name



print(f'Full-data tuned {best_original_name} ({best_config_name}):')

tuned_results

c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Full-data tuned LightGBM (more_capacity):


{'rmsle': 0.36444268804568525,
 'rmse': 506.0430542076931,
 'mae': 197.0072688591496,
 'train_time_sec': 5.73,
 'config': 'more_capacity'}

In [18]:
# Compare the tuned model to the original winner; retain tuning only if it improves RMSLE.

original_rmsle = comparison.loc[comparison['model'] == best_original_name, 'rmsle'].iloc[0]

if tuned_results['rmsle'] < original_rmsle:

    active_model_name = f'{best_original_name} (tuned)'

    active_predictions = tuned_predictions

    comparison = pd.concat(

        [comparison, pd.DataFrame([{'model': active_model_name, **tuned_results}])],

        ignore_index=True,

    )

else:

    active_model_name = best_original_name

    active_predictions = best_original_predictions



val_df = X_val.copy()

val_df['actual'] = y_val.to_numpy()

val_df['predicted'] = active_predictions

val_df['abs_error_sec'] = (val_df['actual'] - val_df['predicted']).abs()

val_df['duration_bucket'] = pd.cut(

    val_df['actual'],

    bins=[0, 300, 600, 1200, 2400, 79200],

    labels=['<5min', '5-10min', '10-20min', '20-40min', '40min+'],

)

print(f'Error segments for: {active_model_name}')

display(val_df.groupby('duration_bucket', observed=True)['abs_error_sec'].mean())

display(val_df.groupby('pickup_hour')['abs_error_sec'].mean())

comparison.sort_values('rmsle').reset_index(drop=True)

Error segments for: LightGBM (tuned)


duration_bucket
<5min       132.691224
5-10min     120.816915
10-20min    172.263654
20-40min    323.380107
40min+      870.103558
Name: abs_error_sec, dtype: float64

pickup_hour
0     172.477087
1     168.710549
2     157.648140
3     161.915221
4     198.027660
5     186.054455
6     154.132649
7     182.745839
8     200.716637
9     209.613058
10    215.867192
11    215.309813
12    219.681374
13    229.612363
14    236.224712
15    235.834273
16    236.534572
17    224.578314
18    199.050585
19    176.708987
20    163.734302
21    163.358043
22    165.997331
23    171.352978
Name: abs_error_sec, dtype: float64

,model,rmsle,rmse,mae,train_time_sec,config
0,LightGBM (tuned),0.364443,506.043054,197.007269,5.73,more_capacity
1,LightGBM,0.369437,506.717963,200.007910,5.66,NaN
2,XGBoost,0.373390,508.540236,203.358139,7.61,NaN
3,Random Forest,0.374704,514.798326,203.876356,144.72,NaN
4,Linear Regression,0.508631,568.868339,277.209412,0.24,NaN


In [19]:
# LGBMRegressor's default importance_type is 'split' at construction time,
# so we need to either pass importance_type='gain' when creating the model,
# or read it off the underlying booster directly after fitting.
gain_importance = lgbm_model.booster_.feature_importance(importance_type="gain")

gain_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": gain_importance,
})
gain_df["importance_norm"] = gain_df["importance"] / gain_df["importance"].sum()
print(gain_df.sort_values("importance_norm", ascending=False))

             feature    importance  importance_norm
6        distance_km  1.560212e+12         0.730916
7        pickup_hour  1.909141e+11         0.089438
4  dropoff_longitude  9.163252e+10         0.042927
5   dropoff_latitude  9.085889e+10         0.042565
2   pickup_longitude  7.966997e+10         0.037323
8   pickup_dayofweek  5.669415e+10         0.026560
3    pickup_latitude  4.772540e+10         0.022358
9         is_weekend  7.462063e+09         0.003496
1    passenger_count  5.524367e+09         0.002588
0          vendor_id  3.903282e+09         0.001829


## Phase 9 — Weighted Blending

Compare equal weighting and validation-RMSLE-based weighting against the tuned LightGBM model.

In [20]:
import numpy as np

# Use the tuned LightGBM model selected in Phase 8.
lgbm_final_preds = final_tuned_model.predict(X_val)

blend_preds_df = pd.DataFrame({
    'rf': rf_val_predictions,
    'xgb': xgb_val_predictions,
    'lgbm': lgbm_final_preds,
    'actual': y_val.to_numpy(),
})
blend_preds_df.head()

,rf,xgb,lgbm,actual
0,849.444873,953.547119,1032.805974,930
1,278.348678,309.892181,290.447030,316
2,1710.894877,1607.156738,1688.704939,2190
3,2266.167496,2328.094727,2111.821970,2119
4,403.628731,396.899628,327.049110,182


In [21]:
def weighted_blend(df: pd.DataFrame, weights: dict) -> np.ndarray:
    """Return a weighted average of the requested model predictions."""
    assert abs(sum(weights.values()) - 1.0) < 1e-6, 'Weights must sum to 1'
    return sum(df[model] * weight for model, weight in weights.items())

equal_weights = {'rf': 1 / 3, 'xgb': 1 / 3, 'lgbm': 1 / 3}
equal_blend_preds = weighted_blend(blend_preds_df, equal_weights)
equal_blend_results = evaluate_all(
    blend_preds_df['actual'].to_numpy(), equal_blend_preds
)

print('Equal-weight blend:')
for metric, value in equal_blend_results.items():
    print(f'  {metric}: {value:.4f}')

Equal-weight blend:
  rmsle: 0.3666
  rmse: 505.0607
  mae: 198.1594


In [22]:
individual_rmsle = {
    model_name: evaluate_all(
        blend_preds_df['actual'].to_numpy(), blend_preds_df[model_name].to_numpy()
    )['rmsle']
    for model_name in ['rf', 'xgb', 'lgbm']
}
print('Individual RMSLE:', individual_rmsle)

inverse_scores = {model_name: 1 / score for model_name, score in individual_rmsle.items()}
total_score = sum(inverse_scores.values())
validation_weights = {
    model_name: score / total_score for model_name, score in inverse_scores.items()
}
print('Validation-based weights:', validation_weights)

weighted_blend_preds = weighted_blend(blend_preds_df, validation_weights)
weighted_blend_results = evaluate_all(
    blend_preds_df['actual'].to_numpy(), weighted_blend_preds
)

print('Validation-weighted blend:')
for metric, value in weighted_blend_results.items():
    print(f'  {metric}: {value:.4f}')

Individual RMSLE: {'rf': 0.37470425496902593, 'xgb': 0.3733900068969842, 'lgbm': 0.36444268804568525}
Validation-based weights: {'rf': 0.32985048089296914, 'xgb': 0.3310114797160984, 'lgbm': 0.3391380393909325}
Validation-weighted blend:
  rmsle: 0.3665
  rmse: 505.0322
  mae: 198.1248


In [23]:
blend_comparison = pd.DataFrame([
    {'approach': 'LightGBM (tuned, standalone)', **tuned_results},
    {'approach': 'Equal-weight blend', **equal_blend_results},
    {'approach': 'Validation-weighted blend', **weighted_blend_results},
])
blend_comparison

,approach,rmsle,rmse,mae,train_time_sec,config
0,"LightGBM (tuned, standalone)",0.364443,506.043054,197.007269,5.73,more_capacity
1,Equal-weight blend,0.366580,505.060748,198.159376,NaN,NaN
2,Validation-weighted blend,0.366531,505.032166,198.124772,NaN,NaN


## Phase 10 — Stacking Ensemble

Train a Linear Regression meta-model on leakage-safe out-of-fold base-model predictions.

In [24]:
# Fast OOF sanity check before the full five-fold process.
sample_indices = X_train.sample(30_000, random_state=42).index
X_sample, y_sample = X_train.loc[sample_indices], y_train.loc[sample_indices]

oof_sample = generate_oof_predictions(
    train_random_forest, X_sample, y_sample, n_splits=3
)
print(oof_sample.shape, X_sample.shape)
print('NaN OOF predictions:', np.isnan(oof_sample).sum())

(30000,) (30000, 10)
NaN OOF predictions: 0


In [25]:
# Full OOF process: each call trains a fresh model for every fold.
print('Generating OOF predictions — Random Forest...')
oof_rf = generate_oof_predictions(train_random_forest, X_train, y_train, n_splits=5)

print('Generating OOF predictions — XGBoost...')
oof_xgb = generate_oof_predictions(train_xgboost, X_train, y_train, n_splits=5)

print('Generating OOF predictions — LightGBM...')
oof_lgbm = generate_oof_predictions(train_lightgbm, X_train, y_train, n_splits=5)

oof_df = pd.DataFrame({'rf': oof_rf, 'xgb': oof_xgb, 'lgbm': oof_lgbm})
oof_df.describe()

Generating OOF predictions — Random Forest...
Generating OOF predictions — XGBoost...
Generating OOF predictions — LightGBM...


c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

,rf,xgb,lgbm
count,1.143380e+06,1.143380e+06,1.143380e+06
mean,8.360520e+02,8.348517e+02,8.350207e+02
std,5.734609e+02,5.622688e+02,5.642344e+02
min,1.192507e+02,-1.010792e+02,-2.837670e+02
25%,4.476609e+02,4.557195e+02,4.530311e+02
50%,6.829898e+02,6.888346e+02,6.833045e+02
75%,1.043076e+03,1.043564e+03,1.044457e+03
max,1.088043e+04,1.651809e+04,1.050873e+04


In [26]:
meta_model = LinearRegression()
meta_model.fit(oof_df, y_train)

print('Meta-learner coefficients (learned weights per base model):')
for name, coefficient in zip(oof_df.columns, meta_model.coef_):
    print(f'  {name}: {coefficient:.4f}')
print(f'  intercept: {meta_model.intercept_:.4f}')

Meta-learner coefficients (learned weights per base model):
  rf: 0.2588
  xgb: 0.2766
  lgbm: 0.4767
  intercept: -10.5258


In [27]:
# These models were fitted on the full X_train split in earlier phases.
val_preds_for_stack = pd.DataFrame({
    'rf': rf_model.predict(X_val),
    'xgb': xgb_model.predict(X_val),
    'lgbm': final_tuned_model.predict(X_val),
})

stacked_preds = meta_model.predict(val_preds_for_stack)
stacked_results = evaluate_all(y_val.to_numpy(), stacked_preds)

print('Stacking ensemble:')
for metric, value in stacked_results.items():
    print(f'  {metric}: {value:.4f}')

Stacking ensemble:
  rmsle: 0.3634
  rmse: 504.5649
  mae: 197.1564


In [28]:
full_ensemble_comparison = pd.DataFrame([
    {'approach': 'LightGBM (tuned, standalone)', **tuned_results},
    {'approach': 'Equal-weight blend', **equal_blend_results},
    {'approach': 'Validation-weighted blend', **weighted_blend_results},
    {'approach': 'Stacking (Linear meta-learner)', **stacked_results},
])
full_ensemble_comparison

,approach,rmsle,rmse,mae,train_time_sec,config
0,"LightGBM (tuned, standalone)",0.364443,506.043054,197.007269,5.73,more_capacity
1,Equal-weight blend,0.366580,505.060748,198.159376,NaN,NaN
2,Validation-weighted blend,0.366531,505.032166,198.124772,NaN,NaN
3,Stacking (Linear meta-learner),0.363372,504.564898,197.156390,NaN,NaN


## Phase 11 — Final Model Selection

Measure artifact size and single-trip inference latency before choosing the production model.

In [29]:
import os
import joblib

def model_size_mb(model, path):
    joblib.dump(model, path)
    size = os.path.getsize(path) / (1024 * 1024)
    os.remove(path)
    return round(size, 3)

sizes = {
    'Random Forest': model_size_mb(rf_model, 'temp_rf.pkl'),
    'XGBoost': model_size_mb(xgb_model, 'temp_xgb.pkl'),
    'LightGBM': model_size_mb(final_tuned_model, 'temp_lgbm.pkl'),
    'Meta-learner': model_size_mb(meta_model, 'temp_meta.pkl'),
}
sizes['Stacking (RF+XGB+LGBM+Meta)'] = (
    sizes['Random Forest']
    + sizes['XGBoost']
    + sizes['LightGBM']
    + sizes['Meta-learner']
)
print('Model sizes (MB):', sizes)

Model sizes (MB): {'Random Forest': 165.451, 'XGBoost': 0.635, 'LightGBM': 1.051, 'Meta-learner': 0.001, 'Stacking (RF+XGB+LGBM+Meta)': 167.13799999999998}


In [30]:
def time_single_prediction(predict_fn, single_row, n_repeats=200):
    times = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        predict_fn(single_row)
        times.append(time.perf_counter() - start)
    return round(np.mean(times) * 1000, 3)

single_row = X_val.iloc[[0]]

def stack_predict(row):
    base_predictions = pd.DataFrame({
        'rf': rf_model.predict(row),
        'xgb': xgb_model.predict(row),
        'lgbm': final_tuned_model.predict(row),
    })
    return meta_model.predict(base_predictions)

latency = {
    'Random Forest': time_single_prediction(rf_model.predict, single_row),
    'XGBoost': time_single_prediction(xgb_model.predict, single_row),
    'LightGBM': time_single_prediction(final_tuned_model.predict, single_row),
    'Stacking (RF+XGB+LGBM+Meta)': time_single_prediction(stack_predict, single_row),
}
print('Single-prediction latency (ms):', latency)

Single-prediction latency (ms): {'Random Forest': np.float64(21.103), 'XGBoost': np.float64(1.705), 'LightGBM': np.float64(0.964), 'Stacking (RF+XGB+LGBM+Meta)': np.float64(36.601)}


## Phase 12 — Serialization and Inference

Save the selected tuned LightGBM model, then verify the reusable inference pipeline independently of a UI.

In [36]:
import os
from pathlib import Path

import joblib

model_path = PROJECT_ROOT / 'models' / 'lightgbm_final.joblib'
model_path.parent.mkdir(exist_ok=True)
joblib.dump(final_tuned_model, model_path)
print(f'Saved model to {model_path}')
print(os.path.getsize(model_path) / (1024 * 1024), 'MB')

Saved model to c:\Users\Asus\Desktop\nyc-taxi-duration-estimator\models\lightgbm_final.joblib
1.0505542755126953 MB


In [32]:
from src.inference.predict import FEATURE_COLUMNS, build_feature_row

test_row = build_feature_row(
    pickup_latitude=40.7580, pickup_longitude=-73.9855,
    dropoff_latitude=40.7484, dropoff_longitude=-73.9857,
    pickup_datetime='2016-03-14 18:30:00',
    passenger_count=2, vendor_id=1,
)
print(test_row)
print('Columns match training:', test_row.columns.tolist() == FEATURE_COLUMNS)

   vendor_id  passenger_count  pickup_longitude  pickup_latitude  \
0          1                2          -73.9855           40.758   

   dropoff_longitude  dropoff_latitude  distance_km  pickup_hour  \
0           -73.9857           40.7484     1.067606           18   

   pickup_dayofweek  is_weekend  
0                 0           0  
Columns match training: True


In [33]:
from src.inference.predict import InvalidTripInputError, load_model, predict

inference_model = load_model(PROJECT_ROOT / 'models' / 'lightgbm_final.joblib')

result = predict(
    inference_model,
    pickup_latitude=40.7580, pickup_longitude=-73.9855,
    dropoff_latitude=40.7484, dropoff_longitude=-73.9857,
    pickup_datetime='2016-03-14 18:30:00',
    passenger_count=2,
)
print(result)

try:
    predict(
        inference_model,
        pickup_latitude=38.5, pickup_longitude=-121.0,
        dropoff_latitude=38.5, dropoff_longitude=-121.0,
        pickup_datetime='2016-03-14 18:30:00',
        passenger_count=2,
    )
except InvalidTripInputError as error:
    print(f'Correctly rejected: {error}')

{'duration_seconds': 545.9, 'duration_formatted': 'Estimated Trip Duration: 9 minutes'}
Correctly rejected: Pickup/dropoff coordinates fall outside the NYC area this model was trained on.


In [34]:
test_cases = [
    ('Short Manhattan trip, evening', dict(
        pickup_latitude=40.7580, pickup_longitude=-73.9855,
        dropoff_latitude=40.7484, dropoff_longitude=-73.9857,
        pickup_datetime='2016-03-14 18:30:00', passenger_count=1,
    )),
    ('Same trip, 3am', dict(
        pickup_latitude=40.7580, pickup_longitude=-73.9855,
        dropoff_latitude=40.7484, dropoff_longitude=-73.9857,
        pickup_datetime='2016-03-14 03:00:00', passenger_count=1,
    )),
    ('Longer trip: Midtown to JFK-area', dict(
        pickup_latitude=40.7580, pickup_longitude=-73.9855,
        dropoff_latitude=40.6650, dropoff_longitude=-73.7834,
        pickup_datetime='2016-03-14 16:00:00', passenger_count=3,
    )),
]

for description, kwargs in test_cases:
    result = predict(inference_model, **kwargs)
    print(f"{description}: {result['duration_formatted']} ({result['duration_seconds']}s)")

Short Manhattan trip, evening: Estimated Trip Duration: 9 minutes (535.6s)
Same trip, 3am: Estimated Trip Duration: 5 minutes (272.0s)
Longer trip: Midtown to JFK-area: Estimated Trip Duration: 1h 7m (4022.1s)
